In [128]:
# --- Citations ---
# While writing this code I have used Hugging Face docs pages as examples.
# Whenever I copied code directly from Hugging Face I have included the link to the docs page where I found it.

# I also watched this video to better understand how LoRA works (https://www.youtube.com/watch?v=XDOSVh9jJiA).
# The example code from the video, found in this GITHUB repo (https://github.com/NeuralNine/youtube-tutorials/tree/main/LoRA%20Fine-Tuning)
# informed elements of this code, I have added comments to such cells.

In [129]:
!pip install torch datasets transformers bitsandbytes rank_bm25 nltk
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForSequenceClassification, GPT2Config, GPT2Tokenizer, TrainingArguments, Trainer, pipeline
from peft import LoraConfig, get_peft_model, PeftModel
import os
from transformers.pipelines.pt_utils import KeyDataset
from sklearn import metrics
from datasets import load_dataset
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt_tab')

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [130]:
dataset = load_dataset("openlifescienceai/medmcqa", "default")
training = dataset['train']
validation = dataset['validation']
testing = dataset['test']

In [131]:
# Code from this Hugging Face page was copied and altered to load the GPT 2 model
# https://huggingface.co/docs/transformers/en/model_doc/gpt2
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilgpt2",
    quantization_config=quantization_config,
    device_map="auto", num_labels=4
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     | 
-------------------------------------------+------------+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED | 
score.weight                               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [132]:
# The writing of these lines was informed by the example code from the LoRA video mentioned in the Citations at the begining of the code
lora = LoraConfig(r=8,target_modules=["c_proj","v_proj"],lora_alpha=16, lora_dropout=0.05)
fine_tuned_model = get_peft_model(model,lora)

In [133]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")
tokenizer.pad_token = " "

In [135]:
# The writing of this function was partially informed by the example code from the LoRA video mentioned in the Citations.
# It has been adapted to fit our MedMCPQ dataset
def tokenize(batch):
    input_text = [f"### Question:\n{question}\n### Option A:\n{optA}\n### Option B:\n{optB}\n### Option C:\n{optC}\n### Option D:\n{optD}"
      for question,optA,optB,optC,optD in zip(batch['question'], batch['opa'], batch['opb'], batch['opc'], batch['opd'])
    ]
    tokenized = tokenizer(input_text, truncation=True, max_length=512, return_tensors='pt', padding="max_length")
    tokenized['labels'] = batch['cop']
    return tokenized

In [136]:
# The writing of this cell was informed by the example code from the LoRA video mentioned in the Citations at the begining of the code
training_tokenized = training.map(tokenize, batched=True)
training_tokenized.set_format(type="torch", device="cuda")
validation_tokenized = validation.map(tokenize, batched=True)
validation_tokenized.set_format(type="torch", device="cuda")
testing_tokenized = testing.map(tokenize, batched=True)
testing_tokenized.set_format(type="torch", device="cuda")

Map:   0%|          | 0/182822 [00:00<?, ? examples/s]

In [ ]:
# The writing of this function was partially informed by the example code from the LoRA video mentioned in the Citations.
# It has been adapted to fit our USMLE dataset
def tokenize_USMLE(batch):
    input_text = [f"### Question:\n{question}\n### Option A:\n{options["A"]}\n### Option B:\n{options["B"]}\n### Option C:\n{options["C"]}\n### Option D:\n{options["D"]}"
      for question,options in zip(batch["question"],batch["options"])
    ]
    tokenized = tokenizer(input_text, truncation=True, max_length=512, return_tensors='pt', padding="max_length")
    tokenized['labels'] = [ord(label) - 65 for label in batch['answer_idx']]
    return tokenized

In [122]:
#USMLE Validation Set Load
dataset = load_dataset("GBaker/MedQA-USMLE-4-options", "default")
US_validation = dataset["test"]
US_validation_tokenized = US_validation.map(tokenize_USMLE, batched=True)
US_validation_tokenized.set_format(type="torch", device="cuda")

In [ ]:
# Fine-tune GPT model using LoRA
training_arguments = TrainingArguments("./medGPT", per_device_train_batch_size=32, num_train_epochs=3, learning_rate=1e-4,
                                       optim="adamw_torch", label_names=["labels"], gradient_accumulation_steps=4, logging_strategy="steps", logging_steps=10)
trainer = Trainer(fine_tuned_model, args=training_arguments, train_dataset=training_tokenized, processing_class=tokenizer, eval_dataset=testing)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss
10,7.296549
20,7.025829
30,6.565238
40,6.104735
50,5.876841
60,5.634737
70,5.598970
80,5.604070
90,5.658645
100,5.558221


TrainOutput(global_step=4287, training_loss=5.472950285121169, metrics={'train_runtime': 10624.0022, 'train_samples_per_second': 51.625, 'train_steps_per_second': 0.404, 'total_flos': 7.209614964267418e+16, 'train_loss': 5.472950285121169, 'epoch': 3.0})

In [ ]:
fine_tuned_model.save_pretrained("./LoRA_GPT2")

NameError: name 'fine_tuned_model' is not defined

In [ ]:
# Load in Fine-Tuned LoRA Model
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True
)
fine_tuned_model = PeftModel.from_pretrained(model, "/content/LoRA_GPT2")

In [75]:
# Tokenizes the corpus
def corpus_tokenizer(batch):
  input_text = [text for text in batch["contents"]]
  # Passage length capped at 100 tokens to limit compute time
  tokenized = tokenizer(input_text, truncation=True, max_length=100, return_tensors='pt', padding="max_length")
  return tokenized

In [76]:
# This website (https://pypi.org/project/rank-bm25/) was referenced when I made code involving BM25 in this cell and the proceeding cells
corpus = load_dataset("MedRAG/textbooks", "default")
contents = corpus["train"]
tokenized_corpus = contents.map(corpus_tokenizer, batched=True)
bm25 = BM25Okapi(tokenized_corpus)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Map:   0%|          | 0/125847 [00:00<?, ? examples/s]

In [113]:
# I used elements of the example from this website (https://www.geeksforgeeks.org/nlp/removing-stop-words-nltk-python/)
# when making my function to remove stop-words
stop_words = set(stopwords.words('english'))

def remove_stop(text):
  tokens = word_tokenize(text)
  filtered_tokens = [token for token in tokens if token not in stop_words]
  return " ".join(filtered_tokens)

def rag_tokenize(batch):
  unformatted_text = [remove_stop(question) for question in batch['question']]
  top_match = [bm25.get_top_n(q, corpus_list, n=1)[0] for q in unformatted_text]
  input_text = [f"### Context:\n {context}\n### Question:\n{question}\n### Option A:\n{optA}\n### Option B:\n{optB}\n### Option C:\n{optC}\n### Option D:\n{optD}"
      for context,question,optA,optB,optC,optD in zip(top_match,batch['question'], batch['opa'], batch['opb'], batch['opc'], batch['opd'])
    ]
  tokenized = tokenizer(input_text, truncation=True, max_length=512, return_tensors='pt', padding="max_length", truncation_side="left")
  tokenized['labels'] = batch['cop']
  return tokenized

def rag_tokenize_USMLE(batch):
  unformatted_text = [remove_stop(question) for question in batch['question']]
  top_match = [bm25.get_top_n(q, corpus_list, n=1)[0] for q in unformatted_text]
  input_text = [f"### Context:\n {context}\n### Question:\n{question}\n### Option A:\n{options["A"]}\n### Option B:\n{options["B"]}\n### Option C:\n{options["C"]}\n### Option D:\n{options["D"]}"
      for context,question,options in zip(top_match,batch["question"],batch["options"])
    ]
  tokenized = tokenizer(input_text, truncation=True, max_length=512, return_tensors='pt', padding="max_length", truncation_side="left")
  tokenized['labels'] = [ord(label) - 65 for label in batch['answer_idx']]
  return tokenized

In [124]:
corpus_list = [text for text in contents["contents"]]
# Perform Tokenization with RAG
test_rag_tokenized = validation.map(rag_tokenize_USMLE, batched=True, batch_size=1)

# USMLE test size was limited due to compute limitations
small_US_validation = US_validation_tokenized.select(range(150))
US_test_rag_tokenized = small_US_validation.map(rag_tokenize_USMLE, batched=True, batch_size=1)
US_test_rag_tokenized.set_format(type="torch", device="cuda")


125847
Dataset({
    features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 150
})


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [115]:
# Function prints the testing accuracy given a model and a testing dataset
from transformers import pipeline
def accuracy(model, dataset):
  with torch.no_grad():
    input_tensor = []
    attention_mask = []
    labels = []
    for input,attention,label in zip(dataset["input_ids"],dataset["attention_mask"],dataset["labels"]):
      input_tensor.append(input)
      attention_mask.append(attention)
      labels.append(label)
    input_tensor = torch.stack(input_tensor)
    attention_mask = torch.stack(attention_mask)
    labels = torch.stack(labels)
    input_chunks = torch.split(input_tensor, 32)
    attention_chunks = torch.split(attention_mask, 32)
    label_chunks = torch.split(labels, 32)
    accuracy = []
    for input_chunk, attention_chunk, label_chunk in zip(input_chunks, attention_chunks, label_chunks):
      results = model(input_ids=input_chunk, attention_mask=attention_chunk).logits.cpu()
      preds = torch.argmax(results,dim=1)
      accuracy.append(metrics.accuracy_score(label_chunk.cpu(), preds))
    print(sum(accuracy)/len(accuracy))


In [125]:
fine_tuned_model.config.pad_token_id = tokenizer.pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id

accuracy(fine_tuned_model, validation_tokenized)
accuracy(model, validation_tokenized)
accuracy(fine_tuned_model, test_rag_tokenized)
accuracy(model, test_rag_tokenized)

accuracy(fine_tuned_model, small_US_validation)
accuracy(model, small_US_validation)
accuracy(fine_tuned_model, US_test_rag_tokenized)
accuracy(model, US_test_rag_tokenized)

0.24204545454545454
0.26704545454545453
0.24204545454545454
0.2767045454545455
